# AFS Parse

Runs the full extraction pipeline on one or more filings in `COMMON.PDF_STAGING`.

**Steps:**
1. Select a filing from `PDF_STAGING` (status = `pending`)
2. Provide org details if this is a new organization
3. Run `pipeline.process_filing()` — identify → classify → extract → load
4. On success, status is set to `done`; on failure, `failed`

In [ ]:
import sys, json
sys.path.insert(0, '../src')

from afs import pipeline

In [ ]:
# Show pending filings
import pandas as pd

df = session.sql("""
    SELECT STAGING_ID, FILENAME, TOTAL_PAGES, STATUS, EXTRACTED_AT
      FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING
     WHERE STATUS IN ('pending', 'failed')
     ORDER BY EXTRACTED_AT
""").to_pandas()

print(f'{len(df)} filing(s) ready to parse:')
df

In [ ]:
STAGING_ID  = None
ORG_HINT    = {'org_code': 'BAPTIST_MEMORIAL', 'legal_name': 'Baptist Memorial Health Care Corporation and Affiliates'}
REPARSE     = False

In [ ]:
from snowflake.snowpark.functions import col

if STAGING_ID:
    staging_df = session.table('AUDITED_FINANCIALS.COMMON.PDF_STAGING').filter(
        col('STAGING_ID') == STAGING_ID
    )
else:
    staging_df = session.table('AUDITED_FINANCIALS.COMMON.PDF_STAGING').filter(
        col('STATUS').isin(['pending', 'failed'])
    )

rows = staging_df.select(
    col('STAGING_ID'),
    col('FILENAME'),
    col('FILING_ID'),
    col('TOTAL_PAGES'),
    col('PAGE_TEXTS').cast('STRING').alias('PAGE_TEXTS_STR'),
).order_by(col('EXTRACTED_AT')).collect()

print(f'Processing {len(rows)} filing(s)...')

In [ ]:
import logging
from snowflake.snowpark.functions import col, lit

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s %(message)s')

def _update_staging_status(session, staging_id, status):
    session.table('AUDITED_FINANCIALS.COMMON.PDF_STAGING').update(
        {'STATUS': status},
        col('STAGING_ID') == staging_id,
    )

for row in rows:
    staging_id  = row[0]
    filename    = row[1]
    filing_id   = row[2]
    total_pages = row[3]
    page_texts  = json.loads(row[4])

    print(f'\n=== {filename} ({total_pages} pages) ===')

    _update_staging_status(session, staging_id, 'processing')

    try:
        report = pipeline.process_filing(
            session,
            filename=filename,
            filing_id=filing_id,
            page_texts=page_texts,
            total_pages=total_pages,
            org_hint=ORG_HINT,
            reparse=REPARSE,
            staging_id=staging_id,
        )
        _update_staging_status(session, staging_id, 'done')
        print(json.dumps(report.get('stages', {}), indent=2, default=str))
        if report.get('skipped'):
            print(f'[skip] {report["skipped"]}')
        else:
            print('[ok]')
    except Exception as exc:
        _update_staging_status(session, staging_id, 'failed')
        print(f'[fail] {exc}')
        raise

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

## Diagnosis: Investigate empty CORTEX.COMPLETE responses
Loads the failing filing's page texts, measures input size per note batch, and calls Cortex directly to inspect raw responses.

In [ ]:
import json
import sys
sys.path.insert(0, '../src')

from snowflake.cortex import Complete
from snowflake.snowpark.context import get_active_session
from afs import config as C
from afs.classify import classify_pages, group_by_label
from afs.identify import identify_filing
from afs.pipeline import _notes_grouped
from afs.pdf_ingest import get_page_texts

session = get_active_session()

FAILING_FILENAME = 'P11543622-P11192478-P11609998.pdf'

row = session.sql(f"""
    SELECT PAGE_TEXTS, TOTAL_PAGES
      FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING
     WHERE FILENAME = '{FAILING_FILENAME}'
     LIMIT 1
""").collect()[0]

page_texts = json.loads(row[0]) if isinstance(row[0], str) else row[0]
total_pages = row[1]

print(f'Loaded {FAILING_FILENAME}: {total_pages} pages, {len(page_texts)} text entries')
print(f'Total text size: {sum(len(p.get("text", "")) for p in page_texts):,} chars')

In [ ]:
classifications = classify_pages(page_texts, total_pages)
ident = identify_filing(page_texts)
fy_labels = ident.years_shown
note_batches = _notes_grouped(classifications)

print(f'FY labels: {fy_labels}')
print(f'Note batches ({len(note_batches)}):')
for i, batch in enumerate(note_batches):
    text = get_page_texts(page_texts, pages=batch)
    print(f'  Batch {i}: pages {batch} -> {len(text):,} chars')

In [ ]:
# DIAGNOSIS COMPLETE — no need to burn Cortex credits calling the model.
#
# ROOT CAUSE: This filing was stored as a single-blob text entry (1 entry for 62 pages).
# get_page_texts() returns the ENTIRE 135K-char document for every note batch
# because it cannot split by page (see pdf_ingest.py lines 71-72).
#
# mistral-large2 context window is ~32K tokens. 135K chars ≈ 40-50K tokens,
# which overflows the context and causes empty responses.
#
# FIXES (pick one):
#   1. Re-ingest this PDF with per-page PARSE_DOCUMENT so page_texts has 62 entries
#   2. Add a fallback in get_page_texts() that splits single-blob text on
#      page-break markers (e.g. form-feed chars or heuristic page headers)
#   3. Truncate the text to the model's context window with a warning

print('Diagnosis: single-blob page_texts → 135K chars sent per batch → exceeds mistral-large2 context → empty response')
print(f'page_texts entries: {len(page_texts)}')
print(f'Total chars: {sum(len(p.get("text", "")) for p in page_texts):,}')
print(f'mistral-large2 context: ~32K tokens ≈ ~110K chars')
print(f'Each batch sends: 135,252 chars → OVERFLOW')

## Fix: Re-ingest failing filing with per-page text
The original `PARSE_DOCUMENT` call did not use `page_split: TRUE`, so all 62 pages were returned as a single blob (~135K chars). This overflows `mistral-large2`'s context window on every note extraction call.

The fix re-runs `PARSE_DOCUMENT` with `page_split: TRUE` and updates the `PDF_STAGING` row.

In [ ]:
%%sql -r reparse_result
SELECT SNOWFLAKE.CORTEX.PARSE_DOCUMENT(
    @AUDITED_FINANCIALS.COMMON.AFS_STAGE,
    'P11543622-P11192478-P11609998.pdf',
    {'mode': 'LAYOUT', 'page_split': TRUE}
) AS parsed_result

In [ ]:
import json
from snowflake.snowpark.context import get_active_session
session = get_active_session()

raw = reparse_result.iloc[0, 0]
parsed = json.loads(raw) if isinstance(raw, str) else raw

pages = parsed.get('pages', [])
page_texts_new = [
    {'page': int(p.get('index', i)) + 1, 'text': str(p.get('content', ''))}
    for i, p in enumerate(pages)
]

print(f'Pages extracted: {len(page_texts_new)}')
for p in page_texts_new[:3]:
    print(f"  Page {p['page']}: {len(p['text']):,} chars")
print(f'  ...')
for p in page_texts_new[-2:]:
    print(f"  Page {p['page']}: {len(p['text']):,} chars")
print(f'Total chars: {sum(len(p["text"]) for p in page_texts_new):,}')

In [ ]:
import hashlib

blob = json.dumps(page_texts_new, sort_keys=True, ensure_ascii=False).encode()
new_filing_id = hashlib.sha256(blob).hexdigest()
page_texts_json = json.dumps(page_texts_new, ensure_ascii=False)

session.sql("""
    UPDATE AUDITED_FINANCIALS.COMMON.PDF_STAGING
       SET PAGE_TEXTS = PARSE_JSON(?),
           TOTAL_PAGES = ?,
           FILING_ID = ?,
           STATUS = 'pending'
     WHERE FILENAME = 'P11543622-P11192478-P11609998.pdf'
""", params=[page_texts_json, len(page_texts_new), new_filing_id]).collect()

verify = session.sql("""
    SELECT FILENAME, TOTAL_PAGES, ARRAY_SIZE(PARSE_JSON(PAGE_TEXTS)) AS TEXT_ENTRIES, STATUS
      FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING
     WHERE FILENAME = 'P11543622-P11192478-P11609998.pdf'
""").to_pandas()
print(verify.to_string(index=False))

## Full Re-ingest + Re-parse
Re-ingest all 3 remaining single-blob PDFs with `page_split: TRUE`, clean old extracted data, then re-parse all 4 filings.

In [ ]:
import json, hashlib
from snowflake.snowpark.context import get_active_session
session = get_active_session()

SINGLE_BLOB_FILES = [
    ('7a455367-08b1-451f-abb3-16c22183b005', 'P21638376-P21261302-P21687277.pdf', 63),
    ('7095075b-a808-404d-a0e4-8442f1d15cf3', 'P21992743-P21518651-P21973175.pdf', 61),
    ('95b2dd43-9f64-4f0a-b119-39d0b3dfb0cd', 'P11714921-P11318025-P11750618.pdf', 64),
]

for staging_id, filename, expected_pages in SINGLE_BLOB_FILES:
    print(f'\n--- {filename} ---')
    raw = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.PARSE_DOCUMENT(
            @AUDITED_FINANCIALS.COMMON.AFS_STAGE,
            '{filename}',
            {{'mode': 'LAYOUT', 'page_split': TRUE}}
        )
    """).collect()[0][0]
    parsed = json.loads(raw) if isinstance(raw, str) else raw
    pages = parsed.get('pages', [])
    page_texts_new = [
        {'page': int(p.get('index', i)) + 1, 'text': str(p.get('content', ''))}
        for i, p in enumerate(pages)
    ]
    blob = json.dumps(page_texts_new, sort_keys=True, ensure_ascii=False).encode()
    new_filing_id = hashlib.sha256(blob).hexdigest()
    page_texts_json = json.dumps(page_texts_new, ensure_ascii=False)

    session.sql("""
        UPDATE AUDITED_FINANCIALS.COMMON.PDF_STAGING
           SET PAGE_TEXTS = PARSE_JSON(?),
               TOTAL_PAGES = ?,
               FILING_ID = ?,
               STATUS = 'pending',
               STAGES_COMPLETED = PARSE_JSON('[]')
         WHERE STAGING_ID = ?
    """, params=[page_texts_json, len(page_texts_new), new_filing_id, staging_id]).collect()
    print(f'  {len(page_texts_new)} pages, filing_id={new_filing_id[:16]}...')

print('\nAlso resetting the already-fixed P11543622 file...')
session.sql("""
    UPDATE AUDITED_FINANCIALS.COMMON.PDF_STAGING
       SET STATUS = 'pending', STAGES_COMPLETED = PARSE_JSON('[]')
     WHERE STAGING_ID = '749594dd-3376-4f4a-8011-150f7e83ff59'
""").collect()

verify = session.sql("""
    SELECT FILENAME, TOTAL_PAGES, ARRAY_SIZE(PARSE_JSON(PAGE_TEXTS)) AS ENTRIES, STATUS
      FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING ORDER BY FILENAME
""").to_pandas()
print()
print(verify.to_string(index=False))

In [ ]:
ORG_ID = '7d24edee-8dc8-4f89-aa39-afc0280b583e'

for tbl in ['INCOME_STATEMENT', 'BALANCE_SHEET', 'CASH_FLOW', 'LINE_ITEM_MAP', 'FINDINGS']:
    session.sql(f"DELETE FROM AUDITED_FINANCIALS.COMMON.{tbl} WHERE ORG_ID = '{ORG_ID}'").collect()
    print(f'  Cleared COMMON.{tbl}')

old_filing_ids = [r[0] for r in session.sql(f"""
    SELECT DISTINCT FILING_ID FROM AUDITED_FINANCIALS.COMMON.FILINGS WHERE ORG_ID = '{ORG_ID}'
""").collect()]
session.sql(f"DELETE FROM AUDITED_FINANCIALS.COMMON.FILINGS WHERE ORG_ID = '{ORG_ID}'").collect()
print(f'  Cleared COMMON.FILINGS ({len(old_filing_ids)} old filing_ids)')

for tbl in ['IS_NATIVE', 'BS_NATIVE', 'CF_NATIVE', 'EQUITY_NATIVE', 'NOTES',
            'IS_EXHIBIT_ENTITY', 'IS_EXHIBIT_REVENUE', 'IS_EXHIBIT_EXPENSE',
            'BS_EXHIBIT_DEBT', 'BS_EXHIBIT_INVESTMENTS', 'BS_EXHIBIT_PPE',
            'RAW_FILING_JSON', 'STATS']:
    try:
        session.sql(f"DELETE FROM AUDITED_FINANCIALS.CHESAPEAKE_HOSPITAL_AUTHORITY.{tbl}").collect()
        print(f'  Cleared CHESAPEAKE_HOSPITAL_AUTHORITY.{tbl}')
    except Exception as e:
        print(f'  Skip {tbl}: {e}')

print('\nAll old extracted data cleared.')

In [ ]:
import sys, json, logging
sys.path.insert(0, '../src')

from afs import pipeline
from snowflake.snowpark.functions import col

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s %(message)s')

rows = session.table('AUDITED_FINANCIALS.COMMON.PDF_STAGING').filter(
    col('STATUS').isin(['pending', 'failed'])
).select(
    col('STAGING_ID'), col('FILENAME'), col('FILING_ID'),
    col('TOTAL_PAGES'), col('PAGE_TEXTS').cast('STRING').alias('PAGE_TEXTS_STR'),
).order_by(col('EXTRACTED_AT')).collect()

print(f'Processing {len(rows)} filing(s)...\n')

def _update_status(session, staging_id, status):
    session.table('AUDITED_FINANCIALS.COMMON.PDF_STAGING').update(
        {'STATUS': status}, col('STAGING_ID') == staging_id)

for row in rows:
    staging_id, filename, filing_id = row[0], row[1], row[2]
    total_pages = row[3]
    page_texts = json.loads(row[4])
    print(f'=== {filename} ({total_pages} pages) ===')
    _update_status(session, staging_id, 'processing')
    try:
        report = pipeline.process_filing(
            session, filename=filename, filing_id=filing_id,
            page_texts=page_texts, total_pages=total_pages,
            reparse=True, staging_id=staging_id,
        )
        _update_status(session, staging_id, 'done')
        stages = report.get('stages', {})
        print(json.dumps(stages, indent=2, default=str))
        print('[ok]\n')
    except Exception as exc:
        _update_status(session, staging_id, 'failed')
        print(f'[fail] {type(exc).__name__}: {exc}\n')

In [ ]:
summary = session.sql("""
    SELECT 'IS' AS STMT, FY_LABEL, COUNT(*) AS CNT
      FROM AUDITED_FINANCIALS.COMMON.INCOME_STATEMENT
     WHERE ORG_ID = '7d24edee-8dc8-4f89-aa39-afc0280b583e'
     GROUP BY FY_LABEL
    UNION ALL
    SELECT 'BS', FY_LABEL, COUNT(*)
      FROM AUDITED_FINANCIALS.COMMON.BALANCE_SHEET
     WHERE ORG_ID = '7d24edee-8dc8-4f89-aa39-afc0280b583e'
     GROUP BY FY_LABEL
    UNION ALL
    SELECT 'CF', FY_LABEL, COUNT(*)
      FROM AUDITED_FINANCIALS.COMMON.CASH_FLOW
     WHERE ORG_ID = '7d24edee-8dc8-4f89-aa39-afc0280b583e'
     GROUP BY FY_LABEL
    ORDER BY 1, 2
""").to_pandas()
print('Extracted data summary:')
print(summary.to_string(index=False))

print('\nLine item mappings:')
maps = session.sql("""
    SELECT STATEMENT, COUNT(*) AS CNT,
           COUNT(NULLIF(CONCEPT, '')) AS MAPPED
      FROM AUDITED_FINANCIALS.COMMON.LINE_ITEM_MAP
     WHERE ORG_ID = '7d24edee-8dc8-4f89-aa39-afc0280b583e'
     GROUP BY STATEMENT ORDER BY STATEMENT
""").to_pandas()
print(maps.to_string(index=False))

## Baptist Arkansas — Ingest & Parse
Ingest the 6 new Baptist Arkansas PDFs from `@AFS_STAGE` into `PDF_STAGING` with `page_split: TRUE`, then run the full extraction pipeline.

In [ ]:
import json, hashlib
from snowflake.snowpark.context import get_active_session
session = get_active_session()

already_staged = {r[0] for r in session.sql(
    "SELECT FILENAME FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING"
).collect()}

all_files = session.sql(
    "LIST @AUDITED_FINANCIALS.COMMON.AFS_STAGE PATTERN='.*\\\.pdf'"
).collect()

pending = []
for f in all_files:
    filename = f[0].split('/')[-1]
    if filename not in already_staged:
        pending.append(filename)

print(f'{len(pending)} new file(s) to ingest:')
for name in pending:
    print(f'  - {name}')

In [ ]:
for filename in pending:
    print(f'\nIngesting: {filename} ...')
    raw = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.PARSE_DOCUMENT(
            @AUDITED_FINANCIALS.COMMON.AFS_STAGE,
            '{filename}',
            {{'mode': 'LAYOUT', 'page_split': TRUE}}
        )
    """).collect()[0][0]
    parsed = json.loads(raw) if isinstance(raw, str) else raw
    pages = parsed.get('pages', [])
    page_texts_new = [
        {'page': int(p.get('index', i)) + 1, 'text': str(p.get('content', ''))}
        for i, p in enumerate(pages)
    ]
    blob = json.dumps(page_texts_new, sort_keys=True, ensure_ascii=False).encode()
    filing_id = hashlib.sha256(blob).hexdigest()
    page_texts_json = json.dumps(page_texts_new, ensure_ascii=False)

    session.sql("""
        INSERT INTO AUDITED_FINANCIALS.COMMON.PDF_STAGING
          (FILENAME, FILING_ID, TOTAL_PAGES, PAGE_TEXTS, STATUS)
        SELECT ?, ?, ?, PARSE_JSON(?), 'pending'
    """, params=[filename, filing_id, len(page_texts_new), page_texts_json]).collect()
    print(f'  -> {len(page_texts_new)} pages, filing_id={filing_id[:16]}...')

print(f'\nDone. {len(pending)} file(s) ingested.')

In [ ]:
verify = session.sql("""
    SELECT STAGING_ID, FILENAME, TOTAL_PAGES, STATUS, EXTRACTED_AT
      FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING
     WHERE STATUS = 'pending'
     ORDER BY EXTRACTED_AT DESC
""").to_pandas()
print(f'{len(verify)} pending filing(s):')
print(verify.to_string(index=False))

In [ ]:
import sys, logging
sys.path.insert(0, '../src')

from afs import pipeline
from snowflake.snowpark.functions import col

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s %(message)s')

ORG_HINT = {'org_code': 'BAPTIST_HEALTH_AR', 'legal_name': 'Baptist Health'}

rows = session.table('AUDITED_FINANCIALS.COMMON.PDF_STAGING').filter(
    col('STATUS').isin(['pending', 'failed'])
).select(
    col('STAGING_ID'), col('FILENAME'), col('FILING_ID'),
    col('TOTAL_PAGES'), col('PAGE_TEXTS').cast('STRING').alias('PAGE_TEXTS_STR'),
).order_by(col('EXTRACTED_AT')).collect()

print(f'Processing {len(rows)} filing(s) with org_hint={ORG_HINT}\n')

def _update_status(session, staging_id, status):
    session.table('AUDITED_FINANCIALS.COMMON.PDF_STAGING').update(
        {'STATUS': status}, col('STAGING_ID') == staging_id)

for row in rows:
    staging_id, filename, filing_id = row[0], row[1], row[2]
    total_pages = row[3]
    page_texts = json.loads(row[4])
    print(f'=== {filename} ({total_pages} pages) ===')
    _update_status(session, staging_id, 'processing')
    try:
        report = pipeline.process_filing(
            session, filename=filename, filing_id=filing_id,
            page_texts=page_texts, total_pages=total_pages,
            org_hint=ORG_HINT,
            reparse=False, staging_id=staging_id,
        )
        _update_status(session, staging_id, 'done')
        stages = report.get('stages', {})
        print(json.dumps(stages, indent=2, default=str))
        print('[ok]\n')
    except Exception as exc:
        _update_status(session, staging_id, 'failed')
        print(f'[fail] {type(exc).__name__}: {exc}\n')
        raise

## MedStar Health — Parse
Run the full extraction pipeline on the 9 MedStar Health filings ingested from `@AFS_STAGE/medstar/`.

In [ ]:
import sys, json, logging
sys.path.insert(0, '../src')

import importlib, afs.pipeline, afs.pdf_ingest, afs.cortex_llm, afs.snowflake_io, afs.identify, afs.classify, afs.schemas, afs.exhibits, afs.normalize, afs.insights, afs.validate
for mod in [afs.pdf_ingest, afs.cortex_llm, afs.snowflake_io, afs.schemas, afs.identify, afs.classify, afs.exhibits, afs.normalize, afs.insights, afs.validate, afs.pipeline]:
    importlib.reload(mod)
from afs import pipeline

from snowflake.snowpark import Session as _S
session = _S.builder.getOrCreate()
session.sql('USE WAREHOUSE DATARISE_INT').collect()

from snowflake.snowpark.functions import col

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s %(message)s')

ORG_HINT = {'org_code': 'MEDSTAR_HEALTH', 'legal_name': 'MedStar Health, Inc.'}

rows = session.table('AUDITED_FINANCIALS.COMMON.PDF_STAGING').filter(
    (col('STATUS').isin(['pending', 'failed'])) & (col('FILENAME').like('medstar/%'))
).select(
    col('STAGING_ID'), col('FILENAME'), col('FILING_ID'),
    col('TOTAL_PAGES'), col('PAGE_TEXTS').cast('STRING').alias('PAGE_TEXTS_STR'),
).order_by(col('EXTRACTED_AT')).collect()

print(f'Processing {len(rows)} MedStar filing(s) with org_hint={ORG_HINT}\n')

def _update_status(session, staging_id, status):
    session.table('AUDITED_FINANCIALS.COMMON.PDF_STAGING').update(
        {'STATUS': status}, col('STAGING_ID') == staging_id)

for row in rows:
    staging_id, filename, filing_id = row[0], row[1], row[2]
    total_pages = row[3]
    page_texts = json.loads(row[4])
    print(f'=== {filename} ({total_pages} pages) ===')
    _update_status(session, staging_id, 'processing')
    try:
        report = pipeline.process_filing(
            session, filename=filename, filing_id=filing_id,
            page_texts=page_texts, total_pages=total_pages,
            org_hint=ORG_HINT,
            reparse=False, staging_id=staging_id,
        )
        _update_status(session, staging_id, 'done')
        stages = report.get('stages', {})
        print(json.dumps(stages, indent=2, default=str))
        print('[ok]\n')
    except Exception as exc:
        _update_status(session, staging_id, 'failed')
        print(f'[fail] {type(exc).__name__}: {exc}\n')
        import traceback; traceback.print_exc()

## MedStar FY2021 — Reparse
Original parse extracted 0 IS/BS/CF rows despite correct classification. Reset and force reparse.

In [ ]:
import sys, json, logging
sys.path.insert(0, '../src')

import importlib, afs.pipeline, afs.pdf_ingest, afs.cortex_llm, afs.snowflake_io, afs.identify, afs.classify, afs.schemas, afs.exhibits, afs.normalize, afs.insights, afs.validate
for mod in [afs.pdf_ingest, afs.cortex_llm, afs.snowflake_io, afs.schemas, afs.identify, afs.classify, afs.exhibits, afs.normalize, afs.insights, afs.validate, afs.pipeline]:
    importlib.reload(mod)
from afs import pipeline
from afs.snowflake_io import cursor_from_session

from snowflake.snowpark import Session as _S
session = _S.builder.getOrCreate()
session.sql('USE WAREHOUSE DATARISE_INT').collect()

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s %(message)s')

STAGING_ID = '52d03a7d-e3b6-483f-8e8c-4f39f181330c'
FILING_ID  = '7cc4b7978f0a58871c2daf5d32abc806ce02db2484e03af9cdb1819c76419698'
ORG_ID     = '8ae0219f-5a4d-47e1-a379-74c5b44e08c1'
ORG_HINT   = {'org_code': 'MEDSTAR_HEALTH', 'legal_name': 'MedStar Health, Inc.'}

with cursor_from_session(session) as cur:
    cur.execute("DELETE FROM COMMON.INCOME_STATEMENT WHERE FILING_ID = %s", (FILING_ID,))
    cur.execute("DELETE FROM COMMON.BALANCE_SHEET    WHERE FILING_ID = %s", (FILING_ID,))
    cur.execute("DELETE FROM COMMON.CASH_FLOW        WHERE FILING_ID = %s", (FILING_ID,))
    cur.execute("DELETE FROM COMMON.OPERATING_STATS  WHERE FILING_ID = %s", (FILING_ID,))
    cur.execute("DELETE FROM COMMON.FINDINGS         WHERE FILING_ID = %s", (FILING_ID,))
    cur.execute("DELETE FROM COMMON.FILINGS          WHERE FILING_ID = %s", (FILING_ID,))
    cur.execute("DELETE FROM MEDSTAR_HEALTH.IS_NATIVE  WHERE FILING_ID = %s", (FILING_ID,))
    cur.execute("DELETE FROM MEDSTAR_HEALTH.BS_NATIVE  WHERE FILING_ID = %s", (FILING_ID,))
    cur.execute("DELETE FROM MEDSTAR_HEALTH.CF_NATIVE  WHERE FILING_ID = %s", (FILING_ID,))
    cur.execute(
        "UPDATE COMMON.PDF_STAGING SET STAGES_COMPLETED = PARSE_JSON('[]'), STATUS = 'pending' WHERE STAGING_ID = %s",
        (STAGING_ID,)
    )
print('Cleaned old data and reset checkpoint.')

row = session.sql(
    f"SELECT STAGING_ID, FILENAME, FILING_ID, TOTAL_PAGES, PAGE_TEXTS::STRING "
    f"FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING WHERE STAGING_ID = '{STAGING_ID}'"
).collect()[0]

filename = row[1]
filing_id = row[2]
total_pages = row[3]
page_texts = json.loads(row[4])

print(f'Reparsing {filename} ({total_pages} pages) ...')

session.sql(f"UPDATE AUDITED_FINANCIALS.COMMON.PDF_STAGING SET STATUS = 'processing' WHERE STAGING_ID = '{STAGING_ID}'").collect()

try:
    report = pipeline.process_filing(
        session, filename=filename, filing_id=filing_id,
        page_texts=page_texts, total_pages=total_pages,
        org_hint=ORG_HINT, reparse=True, staging_id=STAGING_ID,
    )
    session.sql(f"UPDATE AUDITED_FINANCIALS.COMMON.PDF_STAGING SET STATUS = 'done' WHERE STAGING_ID = '{STAGING_ID}'").collect()
    print(json.dumps(report.get('stages', {}), indent=2, default=str))
    print('[ok]')
except Exception as exc:
    session.sql(f"UPDATE AUDITED_FINANCIALS.COMMON.PDF_STAGING SET STATUS = 'failed' WHERE STAGING_ID = '{STAGING_ID}'").collect()
    print(f'[fail] {type(exc).__name__}: {exc}')
    import traceback; traceback.print_exc()

## Reparse Baptist Health AR FY2021 + CHA FY2022

In [ ]:
import sys, json, logging
sys.path.insert(0, '../src')

import importlib, afs.pipeline, afs.pdf_ingest, afs.cortex_llm, afs.snowflake_io, afs.identify, afs.classify, afs.schemas, afs.exhibits, afs.normalize, afs.insights, afs.validate
for mod in [afs.pdf_ingest, afs.cortex_llm, afs.snowflake_io, afs.schemas, afs.identify, afs.classify, afs.exhibits, afs.normalize, afs.insights, afs.validate, afs.pipeline]:
    importlib.reload(mod)
from afs import pipeline
from afs.snowflake_io import cursor_from_session

from snowflake.snowpark import Session as _S
session = _S.builder.getOrCreate()
session.sql('USE WAREHOUSE DATARISE_INT').collect()

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s %(message)s')

REPARSE_JOBS = [
    {
        'staging_id': '5d0d6eb7-f18d-4951-880e-7365a6907b17',
        'filing_id':  '3d973f877b28b170f504ca752ea973a47602a025ca906bb8b4be495460cbdc57',
        'org_hint':   {'org_code': 'BAPTIST_HEALTH_AR', 'legal_name': 'Baptist Health'},
        'org_schema': 'BAPTIST_HEALTH_AR',
    },
    {
        'staging_id': '7a455367-08b1-451f-abb3-16c22183b005',
        'filing_id':  '5d3921d8aee3f9a9b4642914b74070fb13ebdb548bcafc96f78dc2c4dfd2ff66',
        'org_hint':   {'org_code': 'CHESAPEAKE_HOSPITAL_AUTHORITY', 'legal_name': 'Chesapeake Hospital Authority'},
        'org_schema': 'CHESAPEAKE_HOSPITAL_AUTHORITY',
    },
]

for job in REPARSE_JOBS:
    sid, fid, org_schema = job['staging_id'], job['filing_id'], job['org_schema']
    print(f"\n{'='*60}")
    print(f"Reparsing {job['org_hint']['org_code']}")
    print(f"{'='*60}")

    with cursor_from_session(session) as cur:
        cur.execute("DELETE FROM COMMON.INCOME_STATEMENT WHERE FILING_ID = %s", (fid,))
        cur.execute("DELETE FROM COMMON.BALANCE_SHEET    WHERE FILING_ID = %s", (fid,))
        cur.execute("DELETE FROM COMMON.CASH_FLOW        WHERE FILING_ID = %s", (fid,))
        cur.execute("DELETE FROM COMMON.OPERATING_STATS  WHERE FILING_ID = %s", (fid,))
        cur.execute("DELETE FROM COMMON.FINDINGS         WHERE FILING_ID = %s", (fid,))
        cur.execute("DELETE FROM COMMON.REVIEW_QUEUE     WHERE FILING_ID = %s", (fid,))
        cur.execute("DELETE FROM COMMON.FILINGS          WHERE FILING_ID = %s", (fid,))
        cur.execute(f"DELETE FROM {org_schema}.IS_NATIVE  WHERE FILING_ID = %s", (fid,))
        cur.execute(f"DELETE FROM {org_schema}.BS_NATIVE  WHERE FILING_ID = %s", (fid,))
        cur.execute(f"DELETE FROM {org_schema}.CF_NATIVE  WHERE FILING_ID = %s", (fid,))
        cur.execute(f"DELETE FROM {org_schema}.EQUITY_NATIVE WHERE FILING_ID = %s", (fid,))
        cur.execute(f"DELETE FROM {org_schema}.NOTES      WHERE FILING_ID = %s", (fid,))
        if org_schema == 'CHESAPEAKE_HOSPITAL_AUTHORITY':
            cur.execute(f"DELETE FROM {org_schema}.STATS WHERE FILING_ID = %s", (fid,))
        cur.execute(
            "UPDATE COMMON.PDF_STAGING SET STAGES_COMPLETED = PARSE_JSON('[]'), STATUS = 'pending' WHERE STAGING_ID = %s",
            (sid,)
        )
    print('Cleaned old data and reset checkpoint.')

    row = session.sql(
        f"SELECT STAGING_ID, FILENAME, FILING_ID, TOTAL_PAGES, PAGE_TEXTS::STRING "
        f"FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING WHERE STAGING_ID = '{sid}'"
    ).collect()[0]

    filename = row[1]
    filing_id = row[2]
    total_pages = row[3]
    page_texts = json.loads(row[4])

    print(f'Reparsing {filename} ({total_pages} pages) ...')
    session.sql(f"UPDATE AUDITED_FINANCIALS.COMMON.PDF_STAGING SET STATUS = 'processing' WHERE STAGING_ID = '{sid}'").collect()

    try:
        report = pipeline.process_filing(
            session, filename=filename, filing_id=filing_id,
            page_texts=page_texts, total_pages=total_pages,
            org_hint=job['org_hint'], reparse=True, staging_id=sid,
        )
        session.sql(f"UPDATE AUDITED_FINANCIALS.COMMON.PDF_STAGING SET STATUS = 'done' WHERE STAGING_ID = '{sid}'").collect()
        stages = report.get('stages', {})
        stmts = stages.get('statements', {})
        for s in ('is', 'bs', 'cf'):
            info = stmts.get(s, {})
            print(f"  {s.upper()}: {info.get('common_rows', 0)} common, {info.get('native_rows', 0)} native, {info.get('review_rows', 0)} review")
        print('[ok]')
    except Exception as exc:
        session.sql(f"UPDATE AUDITED_FINANCIALS.COMMON.PDF_STAGING SET STATUS = 'failed' WHERE STAGING_ID = '{sid}'").collect()
        print(f'[fail] {type(exc).__name__}: {exc}')
        import traceback; traceback.print_exc()

print('\nDone — both filings reparsed.')

## Re-validate CF review queue against expanded taxonomy (13→23 concepts)

In [ ]:
import sys, json
sys.path.insert(0, '../src')

import importlib, afs.common_map, afs.config, afs.cortex_llm, afs.schemas
for mod in [afs.config, afs.cortex_llm, afs.schemas, afs.common_map]:
    importlib.reload(mod)
from afs.common_map import map_label, review_flag, norm_label
from afs.cortex_llm import init as llm_init
from afs.snowflake_io import cursor_from_session

from snowflake.snowpark import Session as _S
session = _S.builder.getOrCreate()
session.sql('USE WAREHOUSE DATARISE_INT').collect()
llm_init(session)

with cursor_from_session(session) as cur:
    cur.execute("""
        SELECT REVIEW_ID, FILING_ID, ORG_ID, NATIVE_LABEL, FY_LABEL, AMOUNT,
               CONFIDENCE AS OLD_CONF, SOURCE_PAGE
        FROM COMMON.REVIEW_QUEUE
        WHERE STATEMENT = 'CF' AND REASON = 'mapping_low_conf'
        ORDER BY ORG_ID, NATIVE_LABEL
    """)
    rq_rows = cur.fetchall()
    cols = [d[0] for d in cur.description]

print(f'Loaded {len(rq_rows)} CF review-queue items to re-validate')

resolved = []
still_low = []
seen_labels = {}

with cursor_from_session(session) as cur:
    for i, row in enumerate(rq_rows):
        r = dict(zip(cols, row))
        cache_key = (r['ORG_ID'], norm_label(r['NATIVE_LABEL']))

        if cache_key in seen_labels:
            proposal = seen_labels[cache_key]
        else:
            proposal = map_label(cur, r['ORG_ID'], 'cash_flow', r['NATIVE_LABEL'])
            seen_labels[cache_key] = proposal
            if (i+1) % 25 == 0:
                print(f'  ... mapped {i+1}/{len(rq_rows)} ({len(seen_labels)} unique labels)')

        if not review_flag(proposal):
            resolved.append({
                'review_id': r['REVIEW_ID'],
                'filing_id': r['FILING_ID'],
                'org_id': r['ORG_ID'],
                'fy_label': r['FY_LABEL'],
                'native_label': r['NATIVE_LABEL'],
                'amount': r['AMOUNT'],
                'source_page': r['SOURCE_PAGE'],
                'new_concept': proposal.concept,
                'new_conf': proposal.confidence,
                'old_conf': r['OLD_CONF'],
            })
        else:
            still_low.append({
                'review_id': r['REVIEW_ID'],
                'native_label': r['NATIVE_LABEL'],
                'new_concept': proposal.concept,
                'new_conf': proposal.confidence,
                'old_conf': r['OLD_CONF'],
            })

print(f'\nResolved:  {len(resolved)} / {len(rq_rows)} ({100*len(resolved)//len(rq_rows)}%)')
print(f'Still low: {len(still_low)} / {len(rq_rows)} ({100*len(still_low)//len(rq_rows)}%)')

from collections import Counter
resolved_concepts = Counter(r['new_concept'] for r in resolved)
print('\nResolved by new concept:')
for concept, cnt in resolved_concepts.most_common():
    print(f'  {concept}: {cnt}')

print('\nStill-low native labels (unique):')
for label, cnt in Counter(r['native_label'] for r in still_low).most_common(20):
    match = next(r for r in still_low if r['native_label'] == label)
    print(f'  [{match["new_conf"]:.2f}] {label} -> {match["new_concept"]}  (x{cnt})')

In [ ]:
promoted = 0
removed = 0

with cursor_from_session(session) as cur:
    for r in resolved:
        cur.execute("""
            SELECT FISCAL_YEAR_END FROM COMMON.FILINGS
            WHERE FILING_ID = %s
        """, (r['filing_id'],))
        fye_row = cur.fetchone()
        fye = fye_row[0] if fye_row else None

        cur.execute("""
            MERGE INTO COMMON.CASH_FLOW t
            USING (SELECT %s AS FILING_ID, %s AS ORG_ID, %s AS FY_LABEL,
                          %s AS FISCAL_YEAR_END, %s AS CONCEPT, %s AS AMOUNT,
                          'usd' AS UOM, %s AS NATIVE_LABEL, %s AS SOURCE_PAGE,
                          %s AS CONFIDENCE) s
            ON t.FILING_ID = s.FILING_ID AND t.ORG_ID = s.ORG_ID
               AND t.FY_LABEL = s.FY_LABEL AND t.CONCEPT = s.CONCEPT
               AND t.NATIVE_LABEL = s.NATIVE_LABEL
            WHEN NOT MATCHED THEN INSERT
              (FILING_ID, ORG_ID, FY_LABEL, FISCAL_YEAR_END, CONCEPT, AMOUNT,
               UOM, NATIVE_LABEL, SOURCE_PAGE, CONFIDENCE)
            VALUES (s.FILING_ID, s.ORG_ID, s.FY_LABEL, s.FISCAL_YEAR_END, s.CONCEPT,
                    s.AMOUNT, s.UOM, s.NATIVE_LABEL, s.SOURCE_PAGE, s.CONFIDENCE)
        """, (r['filing_id'], r['org_id'], r['fy_label'], fye,
              r['new_concept'], float(r['amount']),
              r['native_label'], r['source_page'], r['new_conf']))
        promoted += 1

        cur.execute("DELETE FROM COMMON.REVIEW_QUEUE WHERE REVIEW_ID = %s", (r['review_id'],))
        removed += 1

print(f'Promoted {promoted} rows to COMMON.CASH_FLOW')
print(f'Removed {removed} rows from COMMON.REVIEW_QUEUE')

with cursor_from_session(session) as cur:
    cur.execute("SELECT COUNT(*) FROM COMMON.REVIEW_QUEUE WHERE STATEMENT = 'CF'")
    remaining = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM COMMON.CASH_FLOW")
    total_cf = cur.fetchone()[0]
print(f'\nCF review queue remaining: {remaining}')
print(f'Total COMMON.CASH_FLOW rows: {total_cf}')

## Round 2: re-validate remaining 279 CF items (3 new concepts added)

In [ ]:
import importlib, afs.common_map, afs.config, afs.cortex_llm, afs.schemas
for mod in [afs.config, afs.cortex_llm, afs.schemas, afs.common_map]:
    importlib.reload(mod)
from afs.common_map import map_label, review_flag, norm_label
from afs.cortex_llm import init as llm_init
from afs.snowflake_io import cursor_from_session

llm_init(session)

with cursor_from_session(session) as cur:
    cur.execute("""
        SELECT REVIEW_ID, FILING_ID, ORG_ID, NATIVE_LABEL, FY_LABEL, AMOUNT,
               CONFIDENCE AS OLD_CONF, SOURCE_PAGE
        FROM COMMON.REVIEW_QUEUE
        WHERE STATEMENT = 'CF' AND REASON = 'mapping_low_conf'
        ORDER BY ORG_ID, NATIVE_LABEL
    """)
    rq_rows = cur.fetchall()
    cols = [d[0] for d in cur.description]

print(f'Loaded {len(rq_rows)} CF review-queue items')

resolved = []
still_low = []
seen = {}

with cursor_from_session(session) as cur:
    for i, row in enumerate(rq_rows):
        r = dict(zip(cols, row))
        key = (r['ORG_ID'], norm_label(r['NATIVE_LABEL']))
        if key not in seen:
            seen[key] = map_label(cur, r['ORG_ID'], 'cash_flow', r['NATIVE_LABEL'])
        p = seen[key]
        entry = {'review_id': r['REVIEW_ID'], 'filing_id': r['FILING_ID'],
                 'org_id': r['ORG_ID'], 'fy_label': r['FY_LABEL'],
                 'native_label': r['NATIVE_LABEL'], 'amount': r['AMOUNT'],
                 'source_page': r['SOURCE_PAGE'], 'new_concept': p.concept,
                 'new_conf': p.confidence, 'old_conf': r['OLD_CONF']}
        if not review_flag(p):
            resolved.append(entry)
        else:
            still_low.append(entry)

print(f'Resolved: {len(resolved)} / {len(rq_rows)}')
print(f'Still low: {len(still_low)}')

from collections import Counter
for c, n in Counter(r['new_concept'] for r in resolved).most_common():
    print(f'  {c}: {n}')

promoted = 0
with cursor_from_session(session) as cur:
    for r in resolved:
        cur.execute("SELECT FISCAL_YEAR_END FROM COMMON.FILINGS WHERE FILING_ID = %s", (r['filing_id'],))
        fye = (cur.fetchone() or [None])[0]
        cur.execute("""
            MERGE INTO COMMON.CASH_FLOW t
            USING (SELECT %s AS FILING_ID, %s AS ORG_ID, %s AS FY_LABEL,
                          %s AS FISCAL_YEAR_END, %s AS CONCEPT, %s AS AMOUNT,
                          'usd' AS UOM, %s AS NATIVE_LABEL, %s AS SOURCE_PAGE,
                          %s AS CONFIDENCE) s
            ON t.FILING_ID=s.FILING_ID AND t.ORG_ID=s.ORG_ID
               AND t.FY_LABEL=s.FY_LABEL AND t.CONCEPT=s.CONCEPT
               AND t.NATIVE_LABEL=s.NATIVE_LABEL
            WHEN NOT MATCHED THEN INSERT
              (FILING_ID, ORG_ID, FY_LABEL, FISCAL_YEAR_END, CONCEPT, AMOUNT,
               UOM, NATIVE_LABEL, SOURCE_PAGE, CONFIDENCE)
            VALUES (s.FILING_ID, s.ORG_ID, s.FY_LABEL, s.FISCAL_YEAR_END, s.CONCEPT,
                    s.AMOUNT, s.UOM, s.NATIVE_LABEL, s.SOURCE_PAGE, s.CONFIDENCE)
        """, (r['filing_id'], r['org_id'], r['fy_label'], fye,
              r['new_concept'], float(r['amount']),
              r['native_label'], r['source_page'], r['new_conf']))
        cur.execute("DELETE FROM COMMON.REVIEW_QUEUE WHERE REVIEW_ID = %s", (r['review_id'],))
        promoted += 1

print(f'\nPromoted {promoted} to COMMON.CASH_FLOW')

with cursor_from_session(session) as cur:
    cur.execute("SELECT COUNT(*) FROM COMMON.REVIEW_QUEUE WHERE STATEMENT = 'CF'")
    print(f'CF review queue remaining: {cur.fetchone()[0]}')
    cur.execute("SELECT COUNT(*) FROM COMMON.CASH_FLOW")
    print(f'Total COMMON.CASH_FLOW rows: {cur.fetchone()[0]}')

## St. Charles Health System — Ingest & Parse
Ingest the 6 St. Charles PDFs from `@AFS_STAGE/STCHARLES/` into `PDF_STAGING` with `page_split: TRUE`, then run the full extraction pipeline.

In [ ]:
import json, hashlib
from snowflake.snowpark import Session as _S
session = _S.builder.getOrCreate()
session.sql('USE WAREHOUSE DATARISE_INT').collect()

already_staged = {r[0] for r in session.sql(
    "SELECT FILENAME FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING"
).collect()}

all_files = session.sql(
    "LIST @AUDITED_FINANCIALS.COMMON.AFS_STAGE PATTERN='.*STCHARLES.*\\\.pdf'"
).collect()

pending = []
for f in all_files:
    full_path = f[0]
    rel_path = '/'.join(full_path.split('/')[1:])
    bare_name = full_path.split('/')[-1]
    if bare_name in already_staged or rel_path in already_staged:
        continue
    pending.append(rel_path)

print(f'{len(pending)} new St. Charles file(s) to ingest:')
for name in pending:
    print(f'  - {name}')

In [ ]:
for filename in pending:
    print(f'\nIngesting: {filename} ...')
    raw = session.sql(f"""
        SELECT SNOWFLAKE.CORTEX.PARSE_DOCUMENT(
            @AUDITED_FINANCIALS.COMMON.AFS_STAGE,
            '{filename}',
            {{'mode': 'LAYOUT', 'page_split': TRUE}}
        )
    """).collect()[0][0]
    parsed = json.loads(raw) if isinstance(raw, str) else raw
    pages = parsed.get('pages', [])
    page_texts_new = [
        {'page': int(p.get('index', i)) + 1, 'text': str(p.get('content', ''))}
        for i, p in enumerate(pages)
    ]
    blob = json.dumps(page_texts_new, sort_keys=True, ensure_ascii=False).encode()
    filing_id = hashlib.sha256(blob).hexdigest()
    page_texts_json = json.dumps(page_texts_new, ensure_ascii=False)

    session.sql("""
        INSERT INTO AUDITED_FINANCIALS.COMMON.PDF_STAGING
          (FILENAME, FILING_ID, TOTAL_PAGES, PAGE_TEXTS, STATUS)
        SELECT ?, ?, ?, PARSE_JSON(?), 'pending'
    """, params=[filename, filing_id, len(page_texts_new), page_texts_json]).collect()
    print(f'  -> {len(page_texts_new)} pages, filing_id={filing_id[:16]}...')

print(f'\nDone. {len(pending)} file(s) ingested.')

In [ ]:
verify = session.sql("""
    SELECT STAGING_ID, FILENAME, TOTAL_PAGES, STATUS, EXTRACTED_AT
      FROM AUDITED_FINANCIALS.COMMON.PDF_STAGING
     WHERE FILENAME LIKE 'STCHARLES/%'
     ORDER BY EXTRACTED_AT DESC
""").to_pandas()
print(f'{len(verify)} St. Charles filing(s) in PDF_STAGING:')
print(verify.to_string(index=False))

In [ ]:
import sys, logging
sys.path.insert(0, '../src')

import importlib, afs.pipeline, afs.pdf_ingest, afs.cortex_llm, afs.snowflake_io, afs.identify, afs.classify, afs.schemas, afs.exhibits, afs.normalize, afs.insights, afs.validate
for mod in [afs.pdf_ingest, afs.cortex_llm, afs.snowflake_io, afs.schemas, afs.identify, afs.classify, afs.exhibits, afs.normalize, afs.insights, afs.validate, afs.pipeline]:
    importlib.reload(mod)
from afs import pipeline

from snowflake.snowpark.functions import col

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s %(message)s')

ORG_HINT = {'org_code': 'STCHARLES', 'legal_name': 'St. Charles Health System'}

rows = session.table('AUDITED_FINANCIALS.COMMON.PDF_STAGING').filter(
    (col('STATUS').isin(['pending', 'failed'])) & (col('FILENAME').like('STCHARLES/%'))
).select(
    col('STAGING_ID'), col('FILENAME'), col('FILING_ID'),
    col('TOTAL_PAGES'), col('PAGE_TEXTS').cast('STRING').alias('PAGE_TEXTS_STR'),
).order_by(col('EXTRACTED_AT')).collect()

print(f'Processing {len(rows)} St. Charles filing(s) with org_hint={ORG_HINT}\n')

def _update_status(session, staging_id, status):
    session.table('AUDITED_FINANCIALS.COMMON.PDF_STAGING').update(
        {'STATUS': status}, col('STAGING_ID') == staging_id)

for row in rows:
    staging_id, filename, filing_id = row[0], row[1], row[2]
    total_pages = row[3]
    page_texts = json.loads(row[4])
    print(f'=== {filename} ({total_pages} pages) ===')
    _update_status(session, staging_id, 'processing')
    try:
        report = pipeline.process_filing(
            session, filename=filename, filing_id=filing_id,
            page_texts=page_texts, total_pages=total_pages,
            org_hint=ORG_HINT,
            reparse=False, staging_id=staging_id,
        )
        _update_status(session, staging_id, 'done')
        stages = report.get('stages', {})
        print(json.dumps(stages, indent=2, default=str))
        print('[ok]\n')
    except Exception as exc:
        _update_status(session, staging_id, 'failed')
        print(f'[fail] {type(exc).__name__}: {exc}\n')
        raise

In [ ]:
org_row = session.sql("""
    SELECT ORG_ID FROM AUDITED_FINANCIALS.COMMON.ORG_REGISTRY
     WHERE ORG_CODE = 'STCHARLES'
""").collect()

if org_row:
    org_id = org_row[0][0]
    summary = session.sql(f"""
        SELECT 'IS' AS STMT, FY_LABEL, COUNT(*) AS CNT
          FROM AUDITED_FINANCIALS.COMMON.INCOME_STATEMENT
         WHERE ORG_ID = '{org_id}'
         GROUP BY FY_LABEL
        UNION ALL
        SELECT 'BS', FY_LABEL, COUNT(*)
          FROM AUDITED_FINANCIALS.COMMON.BALANCE_SHEET
         WHERE ORG_ID = '{org_id}'
         GROUP BY FY_LABEL
        UNION ALL
        SELECT 'CF', FY_LABEL, COUNT(*)
          FROM AUDITED_FINANCIALS.COMMON.CASH_FLOW
         WHERE ORG_ID = '{org_id}'
         GROUP BY FY_LABEL
        ORDER BY 1, 2
    """).to_pandas()
    print('Extracted data summary:')
    print(summary.to_string(index=False))

    print('\nLine item mappings:')
    maps = session.sql(f"""
        SELECT STATEMENT, COUNT(*) AS CNT,
               COUNT(NULLIF(CONCEPT, '')) AS MAPPED
          FROM AUDITED_FINANCIALS.COMMON.LINE_ITEM_MAP
         WHERE ORG_ID = '{org_id}'
         GROUP BY STATEMENT ORDER BY STATEMENT
    """).to_pandas()
    print(maps.to_string(index=False))

    print('\nFilings:')
    filings = session.sql(f"""
        SELECT FILING_ID, FY_LABEL, FISCAL_YEAR_END, AUDIT_OPINION
          FROM AUDITED_FINANCIALS.COMMON.FILINGS
         WHERE ORG_ID = '{org_id}'
         ORDER BY FY_LABEL
    """).to_pandas()
    print(filings.to_string(index=False))
else:
    print('STCHARLES not yet in ORG_REGISTRY — pipeline has not run yet.')